In [1]:

REPO <- paste0(
  "https://raw.githubusercontent.com/",
  "luisfeliperuas18-tech/",
  "PI3-LUISFELIPERUAS-LORENZOLOUZADA-ARTHURDAVINO/main/"
)

source("https://raw.githubusercontent.com/fractalarea/pi3-motor-de-busca/main/motor/motor08.R")

cfg <- cfg_aula()
cfg$acentos <- "translit"

corpus <- read.csv(
  paste0(REPO, "estrutura/codigo/corpus/corpus_baixada_santista.csv"),
  fileEncoding = "UTF-8-BOM",
  stringsAsFactors = FALSE
)

necessidades <- read.csv(
  paste0(REPO, "estrutura/codigo/to%20delete/5-entrega-necessidades.csv"),
  fileEncoding = "UTF-8-BOM",
  stringsAsFactors = FALSE
)

julgamentos <- read.csv(
  paste0(REPO, "estrutura/codigo/corpus/7-qrels_consolidado_resolvidos.csv"),
  stringsAsFactors = FALSE
)

docs <- setNames(corpus$texto, corpus$id)
consultas <- setNames(
  necessidades$texto_consulta,
  necessidades$consulta
)

ix <- montar(docs, cfg)

cat("Documentos:", length(docs), "\n")
cat("Consultas:", length(consultas), "\n")
cat("Julgamentos:", nrow(julgamentos), "\n")


Documentos: 220 
Consultas: 5 
Julgamentos: 86 


In [2]:

pares <- expand.grid(
  consulta = names(consultas),
  id = names(docs),
  stringsAsFactors = FALSE
)

pares$texto_consulta <- unname(
  consultas[pares$consulta]
)

pares$texto_documento <- unname(
  docs[pares$id]
)

stopifnot(
  !anyNA(pares),
  !anyDuplicated(pares[c("consulta", "id")])
)

write.csv(
  pares,
  "pares_consulta_documento.csv",
  row.names = FALSE,
  fileEncoding = "UTF-8"
)

cat("Total de pares:", nrow(pares), "\n")
head(pares[c("consulta", "id")])


Total de pares: 1100 


,consulta,id
,<chr>,<chr>
1,q01,san_a489c65e47
2,q02,san_a489c65e47
3,q03,san_a489c65e47
4,q04,san_a489c65e47
5,q05,san_a489c65e47
6,q01,san_8cc33c2b34


In [3]:

python <- Sys.which("python3")

if (!nzchar(python)) {
  stop("Python 3 não encontrado no ambiente.")
}

instalacao <- system2(
  python,
  c("-m", "pip", "install", "-q", "sentence-transformers")
)

if (instalacao != 0) {
  stop("Não foi possível instalar sentence-transformers.")
}

codigo_python <- c(
  "import pandas as pd",
  "from sentence_transformers import CrossEncoder",
  "",
  "dados = pd.read_csv('pares_consulta_documento.csv')",
  "",
  "modelo = CrossEncoder(",
  "    'cross-encoder/mmarco-mMiniLMv2-L12-H384-v1'",
  ")",
  "",
  "pares = list(zip(",
  "    dados['texto_consulta'].astype(str),",
  "    dados['texto_documento'].astype(str)",
  "))",
  "",
  "dados['escore'] = modelo.predict(",
  "    pares, batch_size=16, show_progress_bar=True",
  ")",
  "",
  "dados[['consulta', 'id', 'escore']].to_csv(",
  "    'ce_escores.csv', index=False",
  ")",
  "",
  "print('Arquivo ce_escores.csv gerado com sucesso!')"
)

writeLines(codigo_python, "gerar_escores.py")

resultado <- system2(
  python,
  "gerar_escores.py"
)

if (resultado != 0) {
  stop("Falha na geração dos escores.")
}

stopifnot(file.exists("ce_escores.csv"))


In [4]:

escores <- read.csv(
  "ce_escores.csv",
  stringsAsFactors = FALSE
)

stopifnot(
  nrow(escores) == nrow(pares),
  !anyNA(escores),
  !anyDuplicated(escores[c("consulta", "id")]),
  setequal(escores$consulta, names(consultas)),
  setequal(escores$id, names(docs))
)

cat("Linhas:", nrow(escores), "\n")
cat("Documentos:", length(unique(escores$id)), "\n")
cat("Consultas:", length(unique(escores$consulta)), "\n")

head(escores)


Linhas: 1100 
Documentos: 220 
Consultas: 5 


,consulta,id,escore
,<chr>,<chr>,<dbl>
1,q01,san_a489c65e47,-0.5368729
2,q02,san_a489c65e47,-2.6792607
3,q03,san_a489c65e47,-6.6629000
4,q04,san_a489c65e47,-1.8870163
5,q05,san_a489c65e47,-5.3908820
6,q01,san_8cc33c2b34,3.8366430


In [5]:

reordenar <- function(rk, k, sc) {
  ids <- names(rk)

  k <- min(k, length(ids))

  if (k == 0) {
    return(ids)
  }

  top <- ids[seq_len(k)]

  nova_ordem <- order(
    -sc[top],
    seq_along(top)
  )

  nova <- top[nova_ordem]

  cauda <- if (k < length(ids)) {
    ids[(k + 1):length(ids)]
  } else {
    character(0)
  }

  c(nova, cauda)
}

ranking_rerank <- function(q, k) {
  bm25 <- ranking_bm25(
    consultas[[q]], ix, cfg
  )

  if (k == 0) {
    return(names(bm25))
  }

  tabela <- escores[
    escores$consulta == q,
  ]

  sc <- setNames(
    tabela$escore,
    tabela$id
  )

  stopifnot(!anyNA(sc[names(bm25)]))

  reordenar(bm25, k, sc)
}

cat("Função criada com sucesso!\n")


Função criada com sucesso!


In [6]:

q <- "q01"
k <- 5

antes <- ranking_rerank(q, 0)
depois <- ranking_rerank(q, k)

comparacao <- data.frame(
  posicao = 1:5,
  BM25 = antes[1:5],
  BM25_Rerank = depois[1:5]
)

print(comparacao)

write.csv(
  comparacao,
  "comparacao_top5.csv",
  row.names = FALSE
)


  posicao           BM25    BM25_Rerank
1       1 san_89a217bd1e san_89a217bd1e
2       2 san_b93b880263 san_787b827e51
3       3 san_31603f8e29 san_b93b880263
4       4 san_787b827e51  sv_758239684f
5       5  sv_758239684f san_31603f8e29


In [7]:

qs <- unique(julgamentos$consulta)

qrels <- lapply(qs, function(q) {
  dados <- julgamentos[
    julgamentos$consulta == q,
  ]

  setNames(dados$grau, dados$documento)
})

names(qrels) <- qs

dev <- c("q01", "q02", "q03")
teste <- c("q04", "q05")

ks <- c(0, 5, 10, 20)

calcular_ap <- function(q, k) {
  ranking <- ranking_rerank(q, k)

  metricas(
    ranking,
    qrels[[q]],
    cfg$limiar
  )[["AP"]]
}

avaliar <- function(consultas_avaliadas) {
  resultado <- sapply(ks, function(k) {
    sapply(consultas_avaliadas, function(q) {
      calcular_ap(q, k)
    })
  })

  colnames(resultado) <- paste0("k", ks)

  data.frame(
    consulta = consultas_avaliadas,
    resultado,
    row.names = NULL
  )
}

resultado_dev <- avaliar(dev)

print(resultado_dev)

map_dev <- colMeans(
  resultado_dev[-1]
)

print(round(map_dev, 3))


  consulta    k0    k5       k10       k20
1      q01 0.750 1.000 1.0000000 1.0000000
2      q02 0.225 0.225 0.2261905 0.1833333
3      q03 1.000 1.000 1.0000000 1.0000000
   k0    k5   k10   k20 
0.658 0.742 0.742 0.728 


In [8]:

melhor_indice <- which.max(map_dev)

melhor_k <- ks[melhor_indice]

cat("Melhor k:", melhor_k, "\n")
cat(
  "MAP no desenvolvimento:",
  round(map_dev[melhor_indice], 3),
  "\n"
)

resultado_teste <- avaliar(teste)

print(resultado_teste)

map_teste <- colMeans(
  resultado_teste[-1]
)

cat("\nMAP no teste:\n")
print(round(map_teste, 3))

comparacao_map <- data.frame(
  k = ks,
  MAP_dev = as.numeric(map_dev),
  MAP_teste = as.numeric(map_teste)
)

print(comparacao_map)

write.csv(
  comparacao_map,
  "comparacao_MAP.csv",
  row.names = FALSE
)


Melhor k: 10 
MAP no desenvolvimento: 0.742 
  consulta        k0 k5 k10 k20
1      q04 1.0000000  1   1   1
2      q05 0.4166667  1   1   1

MAP no teste:
   k0    k5   k10   k20 
0.708 1.000 1.000 1.000 
   k   MAP_dev MAP_teste
1  0 0.6583333 0.7083333
2  5 0.7416667 1.0000000
3 10 0.7420635 1.0000000
4 20 0.7277778 1.0000000


In [9]:

cfg$rerank_k <- melhor_k

conteudo_config <- c(
  "cfg <-",
  capture.output(dput(cfg))
)

writeLines(
  conteudo_config,
  "config.R"
)

source("config.R")

cat("rerank_k =", cfg$rerank_k, "\n")
cat("Arquivo config.R criado!\n")


rerank_k = 10 
Arquivo config.R criado!


In [10]:

relatorio <- c(
  "# Aula 08 - Parte D",
  "",
  "## Projeto Integrador III",
  "",
  "Grupo: Arthur Davino, Lorenzo Louzada e Luis Felipe Ruas",
  "",
  "## Metodologia",
  "",
  "Aplicamos um cross-encoder multilingue aos pares de consultas e documentos.",
  "Os escores foram utilizados para reordenar os primeiros documentos do BM25.",
  "",
  "Foram avaliados k = 0, 5, 10 e 20.",
  "O valor de k foi escolhido com base no maior MAP do desenvolvimento.",
  "",
  "## Resultados",
  "",
  paste("Melhor rerank_k:", melhor_k),
  "",
  "MAP no desenvolvimento:",
  paste(capture.output(print(round(map_dev, 3))), collapse = "\n"),
  "",
  "MAP no teste:",
  paste(capture.output(print(round(map_teste, 3))), collapse = "\n"),
  "",
  "## Configuracao",
  "",
  paste("rerank_k =", melhor_k),
  "",
  "k = 0 significa manter somente o BM25, sem reordenacao."
)

writeLines(
  relatorio,
  "08-entrega.md"
)

cat(paste(relatorio, collapse = "\n"))


# Aula 08 - Parte D

## Projeto Integrador III

Grupo: Arthur Davino, Lorenzo Louzada e Luis Felipe Ruas

## Metodologia

Aplicamos um cross-encoder multilingue aos pares de consultas e documentos.
Os escores foram utilizados para reordenar os primeiros documentos do BM25.

Foram avaliados k = 0, 5, 10 e 20.
O valor de k foi escolhido com base no maior MAP do desenvolvimento.

## Resultados

Melhor rerank_k: 10

MAP no desenvolvimento:
   k0    k5   k10   k20 
0.658 0.742 0.742 0.728 

MAP no teste:
   k0    k5   k10   k20 
0.708 1.000 1.000 1.000 

## Configuracao

rerank_k = 10

k = 0 significa manter somente o BM25, sem reordenacao.

In [11]:

cat("=== RESULTADOS DA AULA 08 ===\n\n")

cat("MAP no desenvolvimento:\n")
print(round(map_dev, 3))

cat("\nMelhor k escolhido:", melhor_k, "\n")

cat("\nMAP no teste:\n")
print(round(map_teste, 3))

cat("\nConfiguração final:\n")
print(cfg$rerank_k)

cat("\nArquivos gerados:\n")
print(list.files(pattern = "ce_escores|config.R|08-entrega|comparacao"))


=== RESULTADOS DA AULA 08 ===

MAP no desenvolvimento:
   k0    k5   k10   k20 
0.658 0.742 0.742 0.728 

Melhor k escolhido: 10 

MAP no teste:
   k0    k5   k10   k20 
0.708 1.000 1.000 1.000 

Configuração final:
[1] 10

Arquivos gerados:
[1] "08-entrega.md"       "ce_escores.csv"      "comparacao_MAP.csv" 
[4] "comparacao_top5.csv" "config.R"           


In [12]:

map_dev <- setNames(
  as.numeric(map_dev),
  paste0("k", ks)
)

melhor_map <- max(map_dev)

candidatos <- which(
  map_dev >= melhor_map - 0.005
)

melhor_k <- min(ks[candidatos])

cfg$rerank_k <- melhor_k

writeLines(
  c("cfg <-", capture.output(dput(cfg))),
  "config.R"
)

source("config.R")

cat("MAP completo:\n")
print(map_dev, digits = 12)

cat("\nRerank escolhido:", cfg$rerank_k, "\n")

cat(
  "MAP Dev:",
  round(map_dev[paste0("k", melhor_k)], 3),
  "\n"
)

cat(
  "MAP Teste:",
  round(map_teste[paste0("k", melhor_k)], 3),
  "\n"
)


MAP completo:
            k0             k5            k10            k20 
0.658333333333 0.741666666667 0.742063492063 0.727777777778 

Rerank escolhido: 5 
MAP Dev: 0.742 
MAP Teste: 1 


In [13]:

relatorio <- c(
  "# Aula 08 - Parte D: Reordenacao Neural",
  "",
  "Grupo: Arthur Davino, Lorenzo Louzada e Luis Felipe Ruas",
  "",
  "## Metodologia",
  "",
  "Foi utilizado um cross-encoder multilingue para calcular",
  "escores dos pares consulta-documento e reordenar o top-k",
  "dos resultados obtidos pelo BM25.",
  "",
  "Corpus: 220 documentos.",
  "Consultas: 5.",
  "Desenvolvimento: q01, q02 e q03.",
  "Teste: q04 e q05.",
  "",
  "## MAP no desenvolvimento",
  "",
  "| k | MAP |",
  "|---|---:|",
  "| 0 | 0.658333 |",
  "| 5 | 0.741667 |",
  "| 10 | 0.742063 |",
  "| 20 | 0.727778 |",
  "",
  "## MAP no teste",
  "",
  "| k | MAP |",
  "|---|---:|",
  "| 0 | 0.708 |",
  "| 5 | 1.000 |",
  "| 10 | 1.000 |",
  "| 20 | 1.000 |",
  "",
  "## Decisao",
  "",
  "O parametro escolhido foi rerank_k = 5.",
  "Embora k = 10 tenha o maior MAP no desenvolvimento,",
  "a diferenca em relacao a k = 5 foi inferior a 0.005.",
  "Foi escolhido k = 5 por apresentar desempenho",
  "praticamente equivalente com menor custo computacional.",
  "",
  "## Conclusao",
  "",
  "O rerank melhorou o MAP do BM25 no desenvolvimento",
  "de 0.658 para 0.742 e no teste de 0.708 para 1.000.",
  "Os resultados indicam melhoria na ordenacao dos documentos",
  "relevantes nas consultas avaliadas.",
  "",
  "Os resultados devem ser interpretados considerando",
  "o numero reduzido de consultas e julgamentos.",
  "",
  "Configuracao final: rerank_k = 5."
)

writeLines(relatorio, "08-entrega.md")

cat("Relatorio atualizado com sucesso!\n")


Relatorio atualizado com sucesso!
